In [4]:
import random
import copy
import math
import time

COLORS = ['Red', 'Blue', 'Green', 'Yellow']
VALUES = [str(i) for i in range(10)] + ['Skip']

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

    def __hash__(self):
        return hash((self.color, self.value))

def generate_full_deck():
    """Generates a simplified UNO deck. 1 of each 0-9 and Skip per color (44 cards)."""
    deck = []
    for color in COLORS:
        for value in VALUES:
            deck.append(Card(color, value))
    return deck

class GameState:
    def __init__(self, player_hands, top_card, deck, current_turn=0):
        self.player_hands = player_hands  # List of 3 lists (for each player)
        self.top_card = top_card
        self.deck = deck  # List of remaining cards
        self.current_turn = current_turn

    def get_prompt_state_representation(self, ai_player_index):
        """Returns the state representation requested in the prompt."""
        opp1 = (ai_player_index + 1) % 3
        opp2 = (ai_player_index + 2) % 3
        return {
            'ai_cards': self.player_hands[ai_player_index],
            'opponent1_cards': self.player_hands[opp1],
            'opponent2_cards': self.player_hands[opp2],
            'top_card': self.top_card,
            'deck': self.deck
        }

    def is_terminal(self):
        """Rule 4: Player wins when cards = 0."""
        for hand in self.player_hands:
            if len(hand) == 0:
                return True
        return False

    def get_winner(self):
        for i, hand in enumerate(self.player_hands):
            if len(hand) == 0:
                return i
        return -1

    def clone(self):
        return GameState(
            [list(hand) for hand in self.player_hands],
            self.top_card,
            list(self.deck),
            self.current_turn
        )

def get_valid_moves(hand, top_card):
    """
    Rule 1: Valid Move
    Same color OR same number.
    Returns a list of Valid Card objects.
    """
    valid_moves = []
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
    return valid_moves

def apply_move(state: GameState, move):
    """
    Rule 4: State Transition.
    If move is a Card, it acts as playing that card.
    If move is 'Draw', it means drawing a card.
    Note: In chance nodes, 'Draw' gets expanded to specific card draws.
    This function handles standard playing or deterministic draws.
    Returns the new state.
    """
    new_state = state.clone()
    p_idx = new_state.current_turn

    if move == 'Draw':
        # Rule 2: Drawing Card. If no valid card, player must draw 1.
        if len(new_state.deck) > 0:
            drawn = new_state.deck.pop(0)
            new_state.player_hands[p_idx].append(drawn)
        # Advance turn
        new_state.current_turn = (new_state.current_turn + 1) % 3
    else:
        # Playing a card
        # Remove card from hand
        try:
            new_state.player_hands[p_idx].remove(move)
        except ValueError:
            pass # Card already not in hand (shouldn't happen in valid simulation)

        new_state.top_card = move

        # Rule 3: Skip Card
        if move.value == 'Skip':
            new_state.current_turn = (new_state.current_turn + 2) % 3
        else:
            new_state.current_turn = (new_state.current_turn + 1) % 3

    return new_state

def apply_chance_draw(state: GameState, drawn_card: Card):
    """
    Applies the outcome of a Chance Node for Expectimax.
    Draws a *specific* drawn_card from the deck.
    """
    new_state = state.clone()
    p_idx = new_state.current_turn

    # Remove this specific card from deck, if possible (to maintain correct probabilities deep in tree)
    try:
        new_state.deck.remove(drawn_card)
    except ValueError:
        pass

    new_state.player_hands[p_idx].append(drawn_card)
    new_state.current_turn = (new_state.current_turn + 1) % 3

    return new_state

def setup_game():
    deck = generate_full_deck()
    random.shuffle(deck)

    player_hands = [[], [], []]
    for _ in range(5): # 5 cards per player
        for i in range(3):
            player_hands[i].append(deck.pop(0))

    # Initial top card must not be a skip for simplicity (or we can just let it be)
    top_card = deck.pop(0)
    while top_card.value == 'Skip':
        deck.append(top_card)
        random.shuffle(deck)
        top_card = deck.pop(0)

    return GameState(player_hands, top_card, deck, current_turn=0)

In [5]:
def evaluate(state: GameState, ai_player_index, strategy='baseline'):
    """
    Evaluation Function.
    Score = 50 - W1(CAI) + W2(Copp) + W3(S)
    """
    winner = state.get_winner()
    if winner == ai_player_index:
        return 1000
    elif winner != -1:
        return -1000 # Opponent won

    ai_hand = state.player_hands[ai_player_index]
    opp1_hand = state.player_hands[(ai_player_index + 1) % 3]
    opp2_hand = state.player_hands[(ai_player_index + 2) % 3]

    c_ai = len(ai_hand)
    c_opp = (len(opp1_hand) + len(opp2_hand)) / 2.0
    s_count = sum(1 for card in ai_hand if card.value == 'Skip')

    if strategy == 'defensive':
        return 50 - 3 * c_ai + 4 * c_opp + 5 * s_count
    elif strategy == 'offensive':
        return 50 - 8 * c_ai + 1 * c_opp + 1 * s_count
    else:
        return 50 - 5 * c_ai + 2 * c_opp + 3 * s_count

def get_best_move_minimax(state: GameState, depth, ai_player_index):
    """
    Returns (scores_dict, best_move) for Minimax explicitly at depth 1.
    """
    valid_moves = get_valid_moves(state.player_hands[state.current_turn], state.top_card)
    if not valid_moves:
        return None, 'Draw'

    best_val = -math.inf
    best_move = None
    scores_dict = {}

    for move in valid_moves:
        child = apply_move(state, move)
        val = minimax(child, depth - 1, ai_player_index)
        scores_dict[move] = val
        if val > best_val:
            best_val = val
            best_move = move

    if best_move is None:
        best_move = sorted(valid_moves, key=lambda x: str(x.value))[0] # Fallback

    return scores_dict, best_move

def minimax(state: GameState, depth, ai_player_index):
    """
    Defensive Minimax used by Player 1.
    Assumes opponents play to minimize the current player's score.
    """
    if state.is_terminal() or depth == 0:
        return evaluate(state, ai_player_index, 'defensive')

    is_maximizing = (state.current_turn == ai_player_index)
    valid_moves = get_valid_moves(state.player_hands[state.current_turn], state.top_card)

    if not valid_moves:
        child = apply_move(state, 'Draw')
        return minimax(child, depth - 1, ai_player_index)

    if is_maximizing:
        best_val = -math.inf
        for move in valid_moves:
            child = apply_move(state, move)
            val = minimax(child, depth - 1, ai_player_index)
            best_val = max(best_val, val)
        return best_val
    else:
        best_val = math.inf
        for move in valid_moves:
            child = apply_move(state, move)
            val = minimax(child, depth - 1, ai_player_index)
            best_val = min(best_val, val)
        return best_val

def get_best_move_expectimax(state: GameState, depth, ai_player_index):
    """
    Returns (expected_scores_dict, best_move) for Expectimax to show all possible decisions at depth 1.
    """
    valid_moves = get_valid_moves(state.player_hands[state.current_turn], state.top_card)
    if not valid_moves:
        return None, 'Draw'

    best_val = -math.inf
    best_move = None
    scores_dict = {}

    for move in valid_moves:
        child = apply_move(state, move)
        val = expectimax(child, depth - 1, ai_player_index)
        scores_dict[move] = val
        if val > best_val:
            best_val = val
            best_move = move

    if best_move is None:
        best_move = valid_moves[0]

    return scores_dict, best_move

def expectimax(state: GameState, depth, ai_player_index):
    """
    Offensive Expectimax used by Player 2.
    Draw actions -> Chance nodes.
    Opponent turns -> Average of random legal moves.
    """
    if state.is_terminal() or depth == 0:
        return evaluate(state, ai_player_index, 'offensive')

    is_maximizing = (state.current_turn == ai_player_index)
    valid_moves = get_valid_moves(state.player_hands[state.current_turn], state.top_card)

    if is_maximizing:
        if not valid_moves:
            return chance_node(state, depth, ai_player_index) # Not decreasing depth for chance node itself

        best_val = -math.inf
        for move in valid_moves:
            child = apply_move(state, move)
            val = expectimax(child, depth - 1, ai_player_index)
            best_val = max(best_val, val)
        return best_val
    else: # Opponent turn
        if not valid_moves:
            return chance_node(state, depth, ai_player_index)

        avg_val = 0
        for move in valid_moves:
            child = apply_move(state, move)
            avg_val += expectimax(child, depth - 1, ai_player_index)
        return avg_val / len(valid_moves)

def chance_node(state: GameState, depth, ai_player_index):
    """
    Probability calculations for drawing cards.
    """
    if not state.deck:
        return evaluate(state, ai_player_index, 'offensive')

    expected_value = 0
    unique_cards = set(state.deck)
    for card in unique_cards:
        prob = state.deck.count(card) / len(state.deck)
        child = apply_chance_draw(state, card)
        expected_value += prob * expectimax(child, depth - 1, ai_player_index)

    return expected_value

In [ ]:
def print_state_and_decisions(player_name, state, scores_dict, move):
    print("\n" + "="*40)
    print(f"Top card: {state.top_card}")
    print(f"{player_name} hand:")
    current_hand = state.player_hands[state.current_turn]
    for card in current_hand:
        print(f"  {card}")

    if scores_dict is not None:
        print(f"{player_name} decision (All possible decisions considered at depth 1):")
        for m, score in scores_dict.items():
            print(f"  Play: {m}")
            print(f"  Expected score: {score:.2f}")

    print(f"\n>>>> Action Taken: {'Draw' if move == 'Draw' else f'Play: {move}'}")
    print("="*40)

def simulate_game(player_3_mode='simulation'):
    print("Starting UNO AI Simulation...")
    print("Player 1 = Minimax (Defensive)")
    print("Player 2 = Expectimax (Offensive)")
    if player_3_mode == 'simulation':
        print("Player 3 = Minimax (Simulation Mode)")
    else:
        print("Player 3 = User (Manual Mode)")

    state = setup_game()
    depth_limit = 3

    players = ["Player 1 (Minimax)", "Player 2 (Expectimax)", "Player 3 (Minimax)"]
    if player_3_mode == 'manual':
        players[2] = "Player 3 (User)"

    # Game Loop
    while not state.is_terminal():
        current_p = state.current_turn
        p_name = players[current_p]

        scores_dict = None
        move = None

        if current_p == 0:
            scores_dict, move = get_best_move_minimax(state, depth_limit, current_p)
        elif current_p == 1:
            scores_dict, move = get_best_move_expectimax(state, depth_limit, current_p)
        elif current_p == 2:
            if player_3_mode == 'simulation':
                scores_dict, move = get_best_move_minimax(state, depth_limit, current_p)
            else:
                valid_moves = get_valid_moves(state.player_hands[current_p], state.top_card)
                print("\n" + "="*40)
                print(f"Top card: {state.top_card}")
                print(f"{p_name} hand:")
                for i, card in enumerate(state.player_hands[current_p]):
                    print(f"  {card}")
                
                if not valid_moves:
                    print(f"\nNo valid moves. You must draw.")
                    move = 'Draw'
                else:
                    print("\nValid moves:")
                    for i, card in enumerate(valid_moves):
                        print(f"  {i}: {card}")
                    
                    while True:
                        try:
                            choice = input("Choose a valid move index to play: ")
                            choice = int(choice)
                            if 0 <= choice < len(valid_moves):
                                move = valid_moves[choice]
                                break
                            else:
                                print("Invalid choice. Try again.")
                        except ValueError:
                            print("Invalid input. Enter a number.")
                
        if current_p != 2 or player_3_mode == 'simulation':
            print_state_and_decisions(p_name, state, scores_dict, move)
        else:
            print(f"\n>>>> Action Taken: {'Draw' if move == 'Draw' else f'Play: {move}'}")
            print("="*40)

        # Apply the chosen move
        state = apply_move(state, move)

    print("\n" + "*"*40)
    winner_idx = state.get_winner()
    print(f"GAME OVER! {players[winner_idx]} WINS!")
    for i, p in enumerate(players):
        print(f"{p} remaining cards: {len(state.player_hands[i])}")
    print("*"*40)

# Uncomment the following lines to run the simulation
# mode = input("Enter Player 3 mode ('manual' or 'simulation'): ").strip().lower()
# if mode not in ['manual', 'simulation']:
#     mode = 'simulation'
# simulate_game(mode)